In [ ]:
%pip install openai python-dotenv

In [ ]:
import os
import re
import json
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

from store import VectorStore

load_dotenv()

# Quick check that the .env values loaded (prints True/False, never the key itself)
for var in ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY",
            "AZURE_OPENAI_API_VERSION", "AZURE_OPENAI_DEPLOYMENT"]:
    print(f"{var}: {'set' if os.getenv(var) else 'MISSING'}")

print("cwd:", Path.cwd())
print("chroma_db exists:", Path("chroma_db").exists())

In [ ]:
from openai import OpenAI

load_dotenv(override=True)   # re-read .env even if values were already loaded

def _temp():
    v = os.getenv("TEMPERATURE", "").strip().lower()
    return None if v in ("", "none") else float(v)

@dataclass(frozen=True)
class Settings:
    endpoint: str = os.getenv("AZURE_OPENAI_ENDPOINT", "")
    api_key: str = os.getenv("AZURE_OPENAI_API_KEY", "")
    deployment: str = os.getenv("AZURE_OPENAI_DEPLOYMENT", "")
    temperature: float | None = _temp()
    max_output_tokens: int = int(os.getenv("MAX_OUTPUT_TOKENS", "1200"))

settings = Settings()
print("endpoint:", settings.endpoint)
print("deployment:", settings.deployment)
print("temperature:", settings.temperature)

if "services.ai.azure.com" in settings.endpoint or "/api/projects" in settings.endpoint:
    print("WARNING: this looks like the PROJECT endpoint. Use the Azure OpenAI endpoint (…openai.azure.com).")

base_url = settings.endpoint.rstrip("/") + "/openai/v1/"
client = OpenAI(base_url=base_url, api_key=settings.api_key)

try:
    resp = client.chat.completions.create(
        model=settings.deployment,
        messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        max_completion_tokens=300,
    )
    print("OK:", repr(resp.choices[0].message.content), resp.usage.total_tokens, "tokens")
except Exception as e:
    print(f"FAIL: {type(e).__name__}: {str(e)[:300]}")

In [ ]:
resource = ""   # from your resouce name from the foundry project URL
candidates = [
    f"https://{resource}.openai.azure.com/openai/v1/",
    f"https://{resource}.services.ai.azure.com/openai/v1/",
]

for url in candidates:
    try:
        c = OpenAI(base_url=url, api_key=settings.api_key)
        r = c.chat.completions.create(
            model=settings.deployment,
            messages=[{"role": "user", "content": "Reply with the single word: ready"}],
            max_completion_tokens=300,
        )
        print("OK  ", url, "->", repr(r.choices[0].message.content))
    except Exception as e:
        print("FAIL", url, "->", type(e).__name__, str(e)[:200])

In [ ]:
from urllib.parse import urlparse

load_dotenv(override=True)
settings = Settings()

def make_base_url(endpoint: str) -> str:
    """Keep only scheme + host, then append /openai/v1/."""
    p = urlparse(endpoint.strip())
    return f"{p.scheme}://{p.netloc}/openai/v1/"

print("endpoint in .env:", settings.endpoint)
base_url = make_base_url(settings.endpoint)
client = OpenAI(base_url=base_url, api_key=settings.api_key)
print("client base_url:", client.base_url)

def probe(label, **extra):
    try:
        r = client.chat.completions.create(
            model=settings.deployment,
            messages=[{"role": "user", "content": "What is 17 * 23? Answer with just the number."}],
            max_completion_tokens=500,
            **extra,
        )
        u = r.usage
        details = getattr(u, "completion_tokens_details", None)
        reasoning = getattr(details, "reasoning_tokens", None) if details else None
        print(f"OK   {label}: {r.choices[0].message.content!r} | "
              f"completion_tokens={u.completion_tokens} reasoning_tokens={reasoning}")
    except Exception as e:
        print(f"FAIL {label}: {type(e).__name__}: {str(e)[:200]}")

probe("no temperature")
probe("temperature=0", temperature=0)
probe("temperature=0.2", temperature=0.2)
probe("reasoning_effort=low", reasoning_effort="low")
probe("reasoning_effort=minimal", reasoning_effort="minimal")

In [ ]:
DISCLAIMER = ("\n\n_This is an informational tool based on the Constitution text "
              "(as amended up to the 27th Amendment, 2025). It is not legal advice._")

@dataclass
class Chunk:
    id: str
    text: str
    meta: dict
    distance: float | None = None
    source: str = "semantic"          # exact | semantic | sibling

@dataclass
class Group:
    key: str                          # article_number, or section/kind for non-articles
    chunks: list[Chunk]
    via: str                          # exact | semantic
    best_distance: float | None = None

    @property
    def meta(self):
        return self.chunks[0].meta

    @property
    def is_article(self):
        return self.meta.get("kind") == "article"

@dataclass
class RAGResult:
    variant: str
    question: str
    answer: str = ""
    detected_articles: list = field(default_factory=list)
    retrieved: list = field(default_factory=list)            # [{key, via, best_distance, chunk_ids}]
    dropped_for_budget: list = field(default_factory=list)
    cited_articles: list = field(default_factory=list)
    cited_not_retrieved: list = field(default_factory=list)  # possible hallucinated citations
    context: str = ""
    model: str = ""
    timings: dict = field(default_factory=dict)
    usage: dict = field(default_factory=dict)
    error: str | None = None
    missing_articles: list = field(default_factory=list)   # detected but not in the store
    config: dict = field(default_factory=dict)             # settings used for this run

    def display(self) -> str:
        return self.answer + DISCLAIMER

    def to_dict(self):
        return asdict(self)

# --- smoke test: load the store and look at one raw result ---
store = VectorStore()
print("chunks in collection:", store.collection.count())

res = store.query("freedom of speech and the press", n_results=5)
print("keys:", list(res.keys()))
for i, m, d in zip(res["ids"][0], res["metadatas"][0], res["distances"][0]):
    print(f"{i:28} article={m.get('article_number')!s:8} kind={m.get('kind'):9} dist={d:.3f}")

exact = store.get_article("19A")
print("\nget_article('19A') keys:", list(exact.keys()))
print("ids:", exact["ids"])

In [ ]:
ART_RE = re.compile(
    r"\bArt(?:icle)?s?\.?\s+(\d+[A-Za-z]{0,3}(?:\s*(?:,|and|&|or)\s*\d+[A-Za-z]{0,3})*)\b",
    re.I,
)
NUM_RE = re.compile(r"\d+[A-Za-z]{0,3}")

def detect_articles(text: str) -> list[str]:
    found = []
    for m in ART_RE.finditer(text):
        for n in NUM_RE.findall(m.group(1)):
            n = n.upper()
            if n not in found:
                found.append(n)
    return found

tests = [
    "What does Article 19A say?",                       # ['19A']
    "Explain Articles 19, 19A and 20",                  # ['19', '19A', '20']
    "What is Art. 175A about?",                         # ['175A']
    "compare article 10 and Article 11",                # ['10', '11']
    "Is article 203cc still in force?",                 # ['203CC']
    "Who appoints the Chief Justice?",                  # []
    "What did the 18th Amendment change?",              # []
    "Tell me about Articles 19-25",                     # range: expect only ['19'] (known gap)
    "Article 6 and high treason",                       # ['6']
    "What does Article 5th say?",                       # edge case: watch this one
]

for t in tests:
    print(f"{detect_articles(t)!s:28} <- {t}c")

In [ ]:
def chunks_from_get(res: dict, source: str) -> list[Chunk]:
    """Convert a flat Chroma get() result into Chunks, ordered by chunk_index."""
    chunks = [Chunk(i, d, m, None, source)
              for i, d, m in zip(res["ids"], res["documents"], res["metadatas"])]
    return sorted(chunks, key=lambda c: c.meta.get("chunk_index", 0))

def exact_lookup(detected: list[str]) -> dict[str, Group]:
    """For each detected article number, fetch all its parts. Missing numbers are skipped."""
    groups, missing = {}, []
    for num in detected:
        chunks = chunks_from_get(store.get_article(num), "exact")
        if chunks:
            groups[num] = Group(num, chunks, "exact", best_distance=-1.0)
        else:
            missing.append(num)
    return groups, missing

for q in ["Article 19A", "Article 175A", "Article 184", "Article 999", "Article 5th"]:
    detected = detect_articles(q)
    groups, missing = exact_lookup(detected)
    print(f"\n{q!r}: detected={detected} missing={missing}")
    for key, g in groups.items():
        m = g.meta
        print(f"  {key}: {len(g.chunks)} chunk(s) | ids={[c.id for c in g.chunks]}")
        print(f"      title={m.get('article_title')!r} omitted={m.get('omitted')} "
              f"pdf_page={m.get('pdf_page')} chunk_count={m.get('chunk_count')}")
        print(f"      parts in order: {[c.meta.get('chunk_index') for c in g.chunks]}")

In [ ]:
def group_key(meta: dict) -> str:
    return meta.get("article_number") or meta.get("section") or meta.get("kind", "unknown")

def semantic_search(query: str, n_results: int = 8, skip_keys: set | None = None,
                    expand: bool = True) -> list[Group]:
    skip_keys = skip_keys or set()
    res = store.query(query, n_results=n_results)       # nested lists -> use [0]

    groups: dict[str, Group] = {}
    for i, d, m, dist in zip(res["ids"][0], res["documents"][0],
                             res["metadatas"][0], res["distances"][0]):
        key = group_key(m)
        if key in skip_keys:                            # already fetched by exact lookup
            continue
        g = groups.setdefault(key, Group(key, [], "semantic", dist))
        g.chunks.append(Chunk(i, d, m, dist, "semantic"))

    for key, g in groups.items():
        if expand and g.meta.get("article_number"):     # articles: pull ALL parts
            full = chunks_from_get(store.get_article(key), "sibling")
            hit_ids = {c.id for c in g.chunks}
            for c in full:
                if c.id in hit_ids:
                    c.source = "semantic"               # mark the parts that were real hits
            if full:
                g.chunks = full
        else:                                           # annex/schedule/preamble: keep hits only
            g.chunks.sort(key=lambda c: c.meta.get("chunk_index", 0))

    return sorted(groups.values(), key=lambda g: g.best_distance)

def show_groups(groups: list[Group]):
    for g in groups:
        srcs = "".join("H" if c.source in ("semantic", "exact") else "s" for c in g.chunks)
        print(f"  {g.key:28} chunks={len(g.chunks):2} best_dist={g.best_distance:.3f}  parts[H=hit,s=sibling]={srcs}")

for q in ["freedom of speech and the press",
          "Who appoints the Chief Justice?",
          "What is the Objectives Resolution?",
          "detention without trial"]:
    print(f"\n{q!r}")
    print(" raw chunk hits:", store.query(q, n_results=8)["ids"][0])
    print(" grouped, no expansion:")
    show_groups(semantic_search(q, expand=False))
    print(" grouped + expanded:")
    show_groups(semantic_search(q))

In [ ]:
def retrieve(question: str, detected: list[str], n_results: int = 8,
             max_groups: int = 5, query_text: str | None = None):
    """Returns (groups, missing). query_text lets variants search with a rewritten query."""
    exact, missing = exact_lookup(detected)
    semantic = semantic_search(query_text or question, n_results=n_results,
                               skip_keys=set(exact))
    groups = list(exact.values()) + semantic[:max_groups]
    return groups, missing

tests = [
    "What does Article 19A say, and how does it relate to freedom of speech?",
    "What is Article 184?",
    "Article 999 and freedom of speech",
    "Who appoints the Chief Justice?",
]

for q in tests:
    detected = detect_articles(q)
    groups, missing = retrieve(q, detected)
    print(f"\n{q!r}\n detected={detected} missing={missing}")
    for g in groups:
        print(f"  {g.via:8} {g.key:30} chunks={len(g.chunks):2} "
              f"chars={sum(len(c.text) for c in g.chunks):6} dist={g.best_distance:.3f}")

In [ ]:
AMEND_RE = re.compile(r"amend|substitut|inserted|repeal|history|changed|omitted|added", re.I)

def label_for(g: Group) -> str:
    m = g.meta
    pages = sorted({c.meta.get("pdf_page") for c in g.chunks if c.meta.get("pdf_page") is not None})
    pg = ""
    if pages:
        pg = f"PDF p. {pages[0]}" + (f"-{pages[-1]}" if len(pages) > 1 else "")
    if g.is_article:
        title = m.get("article_title", "")
        label = f"Article {g.key}" + (f": {title}" if title else "")
        if m.get("omitted"):
            label += " [OMITTED]"
    else:
        label = "Preamble" if m.get("kind") == "preamble" else str(g.key)
    return f"{label} | {pg}".strip(" |")

def build_context(question: str, groups: list[Group], max_chars: int = 24000):
    want_notes = bool(AMEND_RE.search(question))
    parts, dropped, used = [], [], 0
    for g in groups:
        body = "\n".join(c.text for c in g.chunks)
        block = f"[Source {len(parts) + 1}] {label_for(g)}\n{body}"
        if want_notes:
            notes = {c.meta["amendment_notes"] for c in g.chunks if c.meta.get("amendment_notes")}
            if notes:
                block += "\nAmendment notes: " + " || ".join(sorted(notes))
        if used + len(block) > max_chars and parts:      # always keep at least the first source
            dropped.append(g.key)
            continue
        parts.append(block)
        used += len(block)
    return "\n\n---\n\n".join(parts), dropped

tests = [
    "Who appoints the Chief Justice?",
    "What is Article 184?",
    "When was Article 175A inserted?",
    "What is the Objectives Resolution?",
]

for q in tests:
    groups, missing = retrieve(q, detect_articles(q))
    print(f"\n{q!r}")
    for cap in (14000, 24000):
        ctx, dropped = build_context(q, groups, max_chars=cap)
        print(f"  cap={cap}: context_chars={len(ctx):6} dropped={dropped} "
              f"has_notes={'Amendment notes' in ctx}")

# Look at an actual context (small one) to check the formatting
q = "What is Article 184?"
groups, _ = retrieve(q, detect_articles(q))
ctx, _ = build_context(q, groups)
print("\n--- sample context (first 1200 chars) ---")
print(ctx[:1200])

In [ ]:
SYSTEM_PROMPT = """You answer questions about the Constitution of Pakistan using ONLY the numbered sources provided.
Rules:
- The sources reflect the current law, including the 27th Amendment (2025). Treat them as current.
- If the sources do not contain the answer, say so plainly. Do not use outside knowledge.
- Cite every claim like (Article 19) or (Article 175A, PDF p. 120). For the Preamble, Objectives Resolution or Schedules, cite them by name.
- If a source is marked [OMITTED], say that the article has been omitted from the Constitution.
- Amendment notes are page-level footnotes and may refer to a neighbouring article; only use them if they clearly match the article asked about.
- Be concise and plain. Quote sparingly."""

def generate(question: str, context: str):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Sources:\n{context}\n\nQuestion: {question}"},
    ]
    kwargs = dict(model=settings.deployment, messages=messages,
                  max_completion_tokens=settings.max_output_tokens)
    if settings.temperature is not None:
        kwargs["temperature"] = settings.temperature
    resp = client.chat.completions.create(**kwargs)
    choice, u = resp.choices[0], resp.usage
    return (choice.message.content or ""), {
        "prompt_tokens": u.prompt_tokens,
        "completion_tokens": u.completion_tokens,
        "total_tokens": u.total_tokens,
        "finish_reason": choice.finish_reason,
    }

print("temperature setting:", settings.temperature)

for q in ["What is Article 184?",
          "Who appoints the Chief Justice?"]:
    detected = detect_articles(q)
    groups, missing = retrieve(q, detected)
    ctx, dropped = build_context(q, groups)

    t = time.perf_counter()
    answer, usage = generate(q, ctx)
    secs = time.perf_counter() - t

    cited = detect_articles(answer)
    have = {g.key for g in groups if g.key not in dropped}
    print("\n" + "=" * 70)
    print("Q:", q)
    print("-" * 70)
    print(answer)
    print("-" * 70)
    print(f"usage={usage} time={secs:.1f}s")
    print(f"cited={cited} | cited_not_retrieved={[a for a in cited if a not in have]}")

In [ ]:
# aliases so the methods below can call the tested functions without name confusion
_retrieve, _build_context, _generate = retrieve, build_context, generate

class BaseRAG:
    """detect -> retrieve -> build_context -> generate. Variants override single stages."""
    name = "base"

    def __init__(self, n_results=8, max_groups=5, max_context_chars=24000,
                 log_path="logs/rag_runs.jsonl"):
        self.n_results = n_results
        self.max_groups = max_groups
        self.max_context_chars = max_context_chars
        self.log_path = Path(log_path) if log_path else None

    # ---- stages (override these in variants) ----
    def detect(self, question):
        return detect_articles(question)

    def prepare_query(self, question):
        return question                      # hook: rewrite/expand the query for semantic search

    def retrieve(self, question, detected):
        return _retrieve(question, detected, n_results=self.n_results,
                         max_groups=self.max_groups,
                         query_text=self.prepare_query(question))

    def build_context(self, question, groups):
        return _build_context(question, groups, max_chars=self.max_context_chars)

    def generate(self, question, context):
        return _generate(question, context)

    # ---- orchestration ----
    def ask(self, question: str) -> RAGResult:
        r = RAGResult(variant=self.name, question=question, model=settings.deployment)
        r.config = {"n_results": self.n_results, "max_groups": self.max_groups,
                    "max_context_chars": self.max_context_chars,
                    "temperature": settings.temperature,
                    "max_output_tokens": settings.max_output_tokens}
        t0 = time.perf_counter()
        try:
            r.detected_articles = self.detect(question)

            t = time.perf_counter()
            groups, r.missing_articles = self.retrieve(question, r.detected_articles)
            r.timings["retrieve_s"] = round(time.perf_counter() - t, 3)

            r.context, r.dropped_for_budget = self.build_context(question, groups)
            r.retrieved = [{"key": g.key, "via": g.via, "best_distance": g.best_distance,
                            "chunk_ids": [c.id for c in g.chunks]} for g in groups]

            if not groups:
                r.answer = "I couldn't find anything relevant in the Constitution text."
            else:
                t = time.perf_counter()
                r.answer, r.usage = self.generate(question, r.context)
                r.timings["generate_s"] = round(time.perf_counter() - t, 3)

                r.cited_articles = detect_articles(r.answer)
                have = {g.key for g in groups if g.key not in r.dropped_for_budget}
                r.cited_not_retrieved = [a for a in r.cited_articles if a not in have]
        except Exception as e:               # failures go in the log too
            r.error = f"{type(e).__name__}: {e}"
            r.answer = "Sorry, something went wrong while answering."
        r.timings["total_s"] = round(time.perf_counter() - t0, 3)
        self._log(r)
        return r

    def _log(self, r: RAGResult):
        if not self.log_path:
            return
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        with self.log_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")

# ---- test ----
rag = BaseRAG()
for q in ["What does Article 19A say, and how does it relate to freedom of speech?",
          "What is the Objectives Resolution?",
          "When was Article 175A inserted?"]:
    r = rag.ask(q)
    print("\n" + "=" * 70)
    print("Q:", q)
    print(r.display())
    print("-" * 70)
    print("detected:", r.detected_articles, "| missing:", r.missing_articles)
    print("retrieved:", [(g["key"], g["via"]) for g in r.retrieved])
    print("dropped:", r.dropped_for_budget, "| cited:", r.cited_articles,
          "| cited_not_retrieved:", r.cited_not_retrieved)
    print("usage:", r.usage, "| timings:", r.timings, "| error:", r.error)

# verify the log is valid JSONL
lines = Path("logs/rag_runs.jsonl").read_text(encoding="utf-8").splitlines()
last = json.loads(lines[-1])
print(f"\nlog lines: {len(lines)} | last variant={last['variant']!r} | keys={list(last)[:6]}...")

In [ ]:
# aliases so the methods below can call the tested functions without name confusion
_retrieve, _build_context, _generate = retrieve, build_context, generate

class BaseRAG:
    """detect -> retrieve -> build_context -> generate. Variants override single stages."""
    name = "base"

    def __init__(self, n_results=8, max_groups=5, max_context_chars=24000,
                 log_path="logs/rag_runs.jsonl"):
        self.n_results = n_results
        self.max_groups = max_groups
        self.max_context_chars = max_context_chars
        self.log_path = Path(log_path) if log_path else None

    # ---- stages (override these in variants) ----
    def detect(self, question):
        return detect_articles(question)

    def prepare_query(self, question):
        return question                      # hook: rewrite/expand the query for semantic search

    def retrieve(self, question, detected):
        return _retrieve(question, detected, n_results=self.n_results,
                         max_groups=self.max_groups,
                         query_text=self.prepare_query(question))

    def build_context(self, question, groups):
        return _build_context(question, groups, max_chars=self.max_context_chars)

    def generate(self, question, context):
        return _generate(question, context)

    # ---- orchestration ----
    def ask(self, question: str) -> RAGResult:
        r = RAGResult(variant=self.name, question=question, model=settings.deployment)
        r.config = {"n_results": self.n_results, "max_groups": self.max_groups,
                    "max_context_chars": self.max_context_chars,
                    "temperature": settings.temperature,
                    "max_output_tokens": settings.max_output_tokens}
        t0 = time.perf_counter()
        try:
            r.detected_articles = self.detect(question)

            t = time.perf_counter()
            groups, r.missing_articles = self.retrieve(question, r.detected_articles)
            r.timings["retrieve_s"] = round(time.perf_counter() - t, 3)

            r.context, r.dropped_for_budget = self.build_context(question, groups)
            r.retrieved = [{"key": g.key, "via": g.via, "best_distance": g.best_distance,
                            "chunk_ids": [c.id for c in g.chunks]} for g in groups]

            if not groups:
                r.answer = "I couldn't find anything relevant in the Constitution text."
            else:
                t = time.perf_counter()
                r.answer, r.usage = self.generate(question, r.context)
                r.timings["generate_s"] = round(time.perf_counter() - t, 3)

                r.cited_articles = detect_articles(r.answer)
                have = {g.key for g in groups if g.key not in r.dropped_for_budget}
                r.cited_not_retrieved = [a for a in r.cited_articles if a not in have]
        except Exception as e:               # failures go in the log too
            r.error = f"{type(e).__name__}: {e}"
            r.answer = "Sorry, something went wrong while answering."
        r.timings["total_s"] = round(time.perf_counter() - t0, 3)
        self._log(r)
        return r

    def _log(self, r: RAGResult):
        if not self.log_path:
            return
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        with self.log_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")

# ---- test ----
rag = BaseRAG()
for q in ["What does Article 19A say, and how does it relate to freedom of speech?",
          "What is the Objectives Resolution?",
          "When was Article 175A inserted?"]:
    r = rag.ask(q)
    print("\n" + "=" * 70)
    print("Q:", q)
    print(r.display())
    print("-" * 70)
    print("detected:", r.detected_articles, "| missing:", r.missing_articles)
    print("retrieved:", [(g["key"], g["via"]) for g in r.retrieved])
    print("dropped:", r.dropped_for_budget, "| cited:", r.cited_articles,
          "| cited_not_retrieved:", r.cited_not_retrieved)
    print("usage:", r.usage, "| timings:", r.timings, "| error:", r.error)

# verify the log is valid JSONL
lines = Path("logs/rag_runs.jsonl").read_text(encoding="utf-8").splitlines()
last = json.loads(lines[-1])
print(f"\nlog lines: {len(lines)} | last variant={last['variant']!r} | keys={list(last)[:6]}...")

In [ ]:
tests = [
    # (question, my expected article(s), what it tests)
    ("Can the police arrest me without telling me why?",           "10",        "paraphrase, no article"),
    ("Can the government take my land or property?",               "24",        "paraphrase"),
    ("Do children have a right to free education?",                "25A",       "paraphrase"),
    ("Is discrimination based on gender allowed?",                 "25",        "paraphrase"),
    ("Can I be punished for something that wasn't a crime when I did it?", "12", "paraphrase"),
    ("Is slavery or forced labour allowed?",                       "11",        "paraphrase"),
    ("How can the Constitution be changed?",                       "239",       "no article, procedure"),
    ("What happens if the President dies or resigns?",             "49",        "no article, situation"),
    ("Who is the commander of the armed forces?",                  "243",       "27th Amdt may have changed this"),
    ("Article 19 gives freedom of religion, right?",               "19 / 20",   "wrong premise: 19 is speech"),
    ("Kya mujhe bina wajah batay giraftar kiya ja sakta hai?",     "10",        "Roman Urdu, expect retrieval trouble"),
    ("What does the Constitution say about cryptocurrency?",       "none",      "out of scope: should say not found"),
    ("Who is the current Prime Minister of Pakistan?",             "none",      "out of scope, current affairs"),
]

for q, expected, note in tests:
    r = rag.ask(q)
    top = [f"{g['key']}({g['via'][0]}:{g['best_distance']:.2f})" for g in r.retrieved]
    print("\n" + "=" * 78)
    print(f"Q: {q}\n   expected={expected} | tests: {note}")
    print(f"   retrieved: {top}")
    print(f"   cited={r.cited_articles} not_retrieved={r.cited_not_retrieved} "
          f"tokens={r.usage.get('total_tokens')} err={r.error}")
    print("   ANSWER:", r.answer[:450].replace("\n", " "))

In [ ]:
# compact view of the last 13 runs from the log (no new API calls)
rows = [json.loads(l) for l in Path("logs/rag_runs.jsonl").read_text(encoding="utf-8").splitlines()][-13:]
for r in rows:
    keys = [g["key"] for g in r["retrieved"]]
    print(f"{r['question'][:55]:55} | retrieved={keys[:5]} | cited={r['cited_articles']}")

In [ ]:
EVAL_PATH = Path("eval/questions.json")

SEED = [
    # id, question, expected articles (verify these against the text!), category, should_decline
    ("q01", "Can the police arrest me without telling me why?",            ["10"],        "paraphrase", False),
    ("q02", "Can the government take my land or property?",               ["24"],        "paraphrase", False),
    ("q03", "Do children have a right to free education?",                ["25A"],       "paraphrase", False),
    ("q04", "Is discrimination based on gender allowed?",                 ["25"],        "paraphrase", False),
    ("q05", "Can I be punished for something that wasn't a crime when I did it?", ["12"], "paraphrase", False),
    ("q06", "Is slavery or forced labour allowed?",                       ["11"],        "paraphrase", False),
    ("q07", "How can the Constitution be changed?",                       ["239"],       "procedure",  False),
    ("q08", "What happens if the President dies or resigns?",             ["49"],        "situation",  False),
    ("q09", "Who is the commander of the armed forces?",                  ["243"],       "situation",  False),
    ("q10", "Article 19 gives freedom of religion, right?",               ["20"],        "wrong_premise", False),
    ("q11", "Kya mujhe bina wajah batay giraftar kiya ja sakta hai?",     ["10"],        "roman_urdu", False),
    ("q12", "What is the minimum age to become President?",               ["41"],        "paraphrase", False),
    ("q13", "Who appoints the Chief Justice?",                            ["177", "175A"], "paraphrase", False),
    ("q14", "What is the Objectives Resolution?",                         ["2A"],        "non_article", False),
    ("q15", "What does the Constitution say about cryptocurrency?",       [],            "out_of_scope", True),
    ("q16", "Who is the current Prime Minister of Pakistan?",             [],            "out_of_scope", True),
]

if not EVAL_PATH.exists():
    EVAL_PATH.parent.mkdir(parents=True, exist_ok=True)
    data = [{"id": i, "question": q, "expected_articles": e, "category": c,
             "should_decline": d, "verified": False} for i, q, e, c, d in SEED]
    EVAL_PATH.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    print("wrote", EVAL_PATH)

eval_set = json.loads(EVAL_PATH.read_text(encoding="utf-8"))
print(f"{len(eval_set)} questions loaded\n")

def score_retrieval(rag_obj, items=eval_set, ks=(1, 3, 5), verbose=True):
    rows = []
    for it in items:
        if not it["expected_articles"]:
            continue                                  # out-of-scope: judged on the answer, not retrieval
        q = it["question"]
        groups, _ = rag_obj.retrieve(q, rag_obj.detect(q))
        keys = [g.key for g in groups]
        ranks = [keys.index(a) + 1 for a in it["expected_articles"] if a in keys]
        rows.append({"id": it["id"], "category": it["category"], "keys": keys,
                     "first_rank": min(ranks) if ranks else None,
                     "recall": len(ranks) / len(it["expected_articles"])})
    n = len(rows)
    summary = {f"hit@{k}": sum(1 for r in rows if r["first_rank"] and r["first_rank"] <= k) / n for k in ks}
    summary["mean_recall"] = sum(r["recall"] for r in rows) / n
    if verbose:
        for r in rows:
            mark = "MISS" if r["first_rank"] is None else f"rank {r['first_rank']}"
            print(f"{r['id']} {r['category']:14} {mark:7} recall={r['recall']:.1f} top={r['keys'][:5]}")
        print(f"\n{rag_obj.name}: " + " | ".join(f"{k}={v:.2f}" for k, v in summary.items()) + f"  (n={n})")
    return rows, summary

base_rows, base_summary = score_retrieval(rag)

In [ ]:
%load_ext autoreload
%autoreload 2

from base_rag import BaseRAG
from evaluation import load_eval, score_retrieval

eval_set = load_eval()
rag = BaseRAG()
rows, summary = score_retrieval(rag, eval_set)

In [ ]:
rag.ask("What is Article 184?")

In [ ]:
from config import settings, client
from base_rag import BaseRAG
from evaluation import score_retrieval

REWRITE_PROMPT = """Rewrite the user's question as a short search query for finding relevant articles of the Constitution of Pakistan.
- Write in formal English, using the vocabulary a constitution would use (e.g. arrest, detention, property, fundamental rights, appointment, removal).
- If the question is in another language or Roman Urdu, translate it.
- Keep any article numbers exactly as written.
- Output ONLY the search query, no explanation, no quotes."""

class RewriteRAG(BaseRAG):
    name = "rewrite"
    rewrite_prompt = REWRITE_PROMPT          # <-- add this line

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._cache = {}

    def prepare_query(self, question):
        if question in self._cache:
            return self._cache[question]
        try:
            kwargs = dict(
                model=settings.deployment,
                messages=[{"role": "system", "content": REWRITE_PROMPT},
                          {"role": "user", "content": question}],
                max_completion_tokens=100,
            )
            if settings.temperature is not None:
                kwargs["temperature"] = settings.temperature
            text = (client.chat.completions.create(**kwargs).choices[0].message.content or "").strip()
        except Exception as e:
            print(f"rewrite failed ({type(e).__name__}): falling back to original question")
            text = ""
        rewritten = text or question
        self._cache[question] = rewritten
        return rewritten

# ---- compare against the baseline (retrieval only; ~14 tiny LLM calls for the rewrites) ----
rw = RewriteRAG()
base_rows, base_summary = score_retrieval(rag, eval_set, verbose=False)
rw_rows, rw_summary = score_retrieval(rw, eval_set, verbose=False)

print("rewritten queries:")
for it in eval_set:
    if it["expected_articles"]:
        print(f"  {it['id']}: {rw.prepare_query(it['question'])!r}")

print("\nrank per question (base -> rewrite):")
fmt = lambda r: "MISS" if r["first_rank"] is None else str(r["first_rank"])
for b, w in zip(base_rows, rw_rows):
    flag = "" if fmt(b) == fmt(w) else ("  <-- better" if (w["first_rank"] or 99) < (b["first_rank"] or 99) else "  <-- WORSE")
    print(f"  {b['id']} {b['category']:14} {fmt(b):>4} -> {fmt(w):>4}{flag}")

print("\nbase   :", " | ".join(f"{k}={v:.2f}" for k, v in base_summary.items()))
print("rewrite:", " | ".join(f"{k}={v:.2f}" for k, v in rw_summary.items()))

In [ ]:
REWRITE_PROMPT_NONUM = REWRITE_PROMPT.replace(
    "- Keep any article numbers exactly as written.",
    "- Keep article numbers ONLY if the user wrote them. Never add article numbers yourself.")
assert REWRITE_PROMPT_NONUM != REWRITE_PROMPT, "replace() matched nothing"

class RewriteNoNumRAG(RewriteRAG):
    name = "rewrite_nonum"
    rewrite_prompt = REWRITE_PROMPT_NONUM

rw = RewriteRAG()            # re-create so it picks up the new class attribute
rn = RewriteNoNumRAG()
rn_rows, rn_summary = score_retrieval(rn, eval_set, verbose=False)
rw_rows, rw_summary = score_retrieval(rw, eval_set, verbose=False)

print("rewritten queries (no-numbers variant):")
for it in eval_set:
    if it["expected_articles"]:
        print(f"  {it['id']}: {rn.prepare_query(it['question'])!r}")

fmt = lambda r: "MISS" if r["first_rank"] is None else str(r["first_rank"])
print("\nrank per question:        base  rewrite  nonum")
for b, w, n in zip(base_rows, rw_rows, rn_rows):
    print(f"  {b['id']} {b['category']:14} {fmt(b):>4} {fmt(w):>7} {fmt(n):>6}")

for label, s in [("base", base_summary), ("rewrite", rw_summary), ("rewrite_nonum", rn_summary)]:
    print(f"{label:14}:", " | ".join(f"{k}={v:.2f}" for k, v in s.items()))

In [ ]:
from base_rag import ART_RE, NUM_RE, detect_articles

def strip_new_article_numbers(rewritten: str, original: str) -> str:
    """Remove article numbers the user did not write from a rewritten query."""
    allowed = set(detect_articles(original))
    def repl(m):
        nums = [n.upper() for n in NUM_RE.findall(m.group(1))]
        return m.group(0) if all(n in allowed for n in nums) else ""
    out = ART_RE.sub(repl, rewritten)
    # leftover bare numbers like "42 43" or "175A"
    out = re.sub(r"\b\d{1,3}[A-Za-z]{0,3}\b",
                 lambda m: m.group(0) if m.group(0).upper() in allowed else "", out)
    return re.sub(r"\s+", " ", out).strip()

# quick unit checks (no LLM)
assert strip_new_article_numbers("slavery article 11 Constitution", "Is slavery allowed?") == "slavery Constitution"
assert strip_new_article_numbers("President article 41 42 43 vacancy", "President dies?") == "President vacancy"
assert "19" in strip_new_article_numbers("Article 19 freedom", "Is Article 19 about religion?")

class RewriteStrictRAG(RewriteRAG):
    name = "rewrite_strict"
    cache_path = Path("eval/rewrite_cache.json")

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._disk = (json.loads(self.cache_path.read_text(encoding="utf-8"))
                      if self.cache_path.exists() else {})

    def prepare_query(self, question):
        if question not in self._disk:
            raw = super().prepare_query(question)
            if raw != question:                      # don't cache failures
                self._disk[question] = raw
                self.cache_path.parent.mkdir(parents=True, exist_ok=True)
                self.cache_path.write_text(json.dumps(self._disk, indent=2, ensure_ascii=False),
                                           encoding="utf-8")
            else:
                return question
        return strip_new_article_numbers(self._disk[question], question)

rs = RewriteStrictRAG()
rs_rows, rs_summary = score_retrieval(rs, eval_set, verbose=False)

print("strict queries:")
for it in eval_set:
    if it["expected_articles"]:
        print(f"  {it['id']}: {rs.prepare_query(it['question'])!r}")

print("\nrank per question:        base  rewrite  strict")
for b, w, s in zip(base_rows, rw_rows, rs_rows):
    print(f"  {b['id']} {b['category']:14} {fmt(b):>4} {fmt(w):>7} {fmt(s):>7}")

for label, s in [("base", base_summary), ("rewrite", rw_summary), ("rewrite_strict", rs_summary)]:
    print(f"{label:14}:", " | ".join(f"{k}={v:.2f}" for k, v in s.items()))

# end-to-end: do the answers improve for the two former misses?
for qid in ("q02", "q11"):
    it = next(i for i in eval_set if i["id"] == qid)
    r = rs.ask(it["question"])
    print(f"\n{qid}: {it['question']}\n  retrieved={[g['key'] for g in r.retrieved]}\n  {r.answer[:400]}")

In [ ]:
%load_ext autoreload
%autoreload 2

from variants import RewriteStrictRAG, _selftest
from evaluation import score_retrieval, load_eval
from base_rag import BaseRAG

_selftest()
eval_set = load_eval()

rows, summary = score_retrieval(RewriteStrictRAG(), eval_set, verbose=False)
print("rewrite_strict:", " | ".join(f"{k}={v:.2f}" for k, v in summary.items()))

# latency/cost of the extra rewrite call (needs one real ask; rewrite is cached, so use a new question)
r = RewriteStrictRAG().ask("Can a minister be removed from office?")
print(r.extra["query_used"], "|", r.timings, "|", r.usage.get("total_tokens"))

In [ ]:
NEW = [
    # id, question, expected, category, should_decline
    ("q17", "What does Article 63 say?",                                   ["63"],   "named_article", False),
    ("q18", "What is Article 6 about?",                                    ["6"],    "named_article", False),
    ("q19", "What did Article 96 say?",                                    ["96"],   "omitted",       False),
    ("q20", "Can Parliament remove the President from office?",            ["47"],   "situation",     False),
    ("q21", "Who is eligible to be a member of the National Assembly?",    ["62"],   "paraphrase",    False),
    ("q22", "What is the official language of Pakistan?",                  ["251"],  "paraphrase",    False),
    ("q23", "Is Islam the state religion?",                                ["2"],    "paraphrase",    False),
    ("q24", "Do I have the right to a fair trial?",                        ["10A"],  "paraphrase",    False),
    ("q25", "Can a person be forced to testify against himself?",          ["13"],   "paraphrase",    False),
    ("q26", "How are members of the Senate elected?",                      ["59"],   "procedure",     False),
    ("q27", "What does the Constitution say about minorities?",            ["36"],   "paraphrase",    False),
    ("q28", "What happens when an emergency is declared?",                 ["232"],  "situation",     False),
    ("q29", "What oath does the Prime Minister take?",                     ["Third Schedule"], "schedule", False),
    ("q30", "Who conducts elections in Pakistan?",                         ["218"],  "paraphrase",    False),
    ("q31", "Kya Pakistan ka sarkari mazhab Islam hai?",                   ["2"],    "roman_urdu",    False),
    ("q32", "At what age must a Supreme Court judge retire?",              ["179"],  "paraphrase",    False),
    ("q33", "What are the current income tax rates in Pakistan?",          [],       "out_of_scope",  True),
    ("q34", "Can you advise me on my divorce case?",                       [],       "out_of_scope",  True),
]

eval_set = load_eval()
have = {it["id"] for it in eval_set}
added = [{"id": i, "question": q, "expected_articles": e, "category": c,
          "should_decline": d, "verified": False, "split": "heldout"}
         for i, q, e, c, d in NEW if i not in have]
for it in eval_set:
    it.setdefault("split", "dev")           # tag the original 16 as the dev set
eval_set += added
EVAL_PATH.write_text(json.dumps(eval_set, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{len(added)} added, {len(eval_set)} total")

# ---- verification helper: print what each expected article actually says ----
store = rag.store
for it in eval_set:
    if it.get("split") != "heldout" or not it["expected_articles"]:
        continue
    print(f"\n{it['id']}: {it['question']}")
    for a in it["expected_articles"]:
        res = store.get_article(a)
        if res["ids"]:
            m, doc = res["metadatas"][0], res["documents"][0]
            body = doc.split("\n\n", 1)[-1][:220].replace("\n", " ")
            print(f"   {a}: {m.get('article_title')!r} omitted={m.get('omitted')}\n      {body}")
        else:
            print(f"   {a}: no article with that number (schedule/annex? check by hand)")

In [ ]:
# 1) check the Third Schedule chunk for the PM's oath
res = store.collection.get(where={"section": "Third Schedule"})
print("Third Schedule chunks:", res["ids"])
for i, d in zip(res["ids"], res["documents"]):
    print(i, "| mentions 'Prime Minister':", "Prime Minister" in d)

# 2) mark as verified the held-out items you've checked (edit this list!)
VERIFIED = {"q17","q18","q19","q20","q21","q22","q23","q24","q25","q26",
            "q27","q28","q30","q31","q32"}          # add "q29" once the schedule check looks right
for it in eval_set:
    if it["id"] in VERIFIED:
        it["verified"] = True
EVAL_PATH.write_text(json.dumps(eval_set, indent=2, ensure_ascii=False), encoding="utf-8")

# 3) score base vs strict on the HELD-OUT split only
heldout = [it for it in eval_set if it.get("split") == "heldout"]
base = BaseRAG()
strict = RewriteStrictRAG()

b_rows, b_sum = score_retrieval(base, heldout, verbose=False)
s_rows, s_sum = score_retrieval(strict, heldout, verbose=False)

fmt = lambda r: "MISS" if r["first_rank"] is None else str(r["first_rank"])
print("\nrank per held-out question:   base  strict")
for b, s in zip(b_rows, s_rows):
    flag = "" if fmt(b) == fmt(s) else ("  <-- better" if (s["first_rank"] or 99) < (b["first_rank"] or 99) else "  <-- WORSE")
    print(f"  {b['id']} {b['category']:14} {fmt(b):>4} {fmt(s):>7}{flag}")

for label, s in [("base", b_sum), ("rewrite_strict", s_sum)]:
    print(f"{label:14}:", " | ".join(f"{k}={v:.2f}" for k, v in s.items()), f"(n={len(b_rows)})")

In [ ]:
URDU_HINTS = {"kya","kia","kay","ke","ki","ka","hai","hain","mujhe","bina","wajah","nahi","nahin",
              "sakta","sakti","sakte","mein","aap","koi","agar","liye","batay","bataye","kaun","kon",
              "kab","kahan","kyun","kyu","hota","hoti","hote","kar","karna","kiya","giraftar",
              "mazhab","sarkari","hukumat","qanoon"}

def is_roman_urdu(text: str) -> bool:
    if re.search(r"[\u0600-\u06FF]", text):          # Urdu/Arabic script
        return True
    words = set(re.findall(r"[a-z]+", text.lower()))
    return len(words & URDU_HINTS) >= 2

# sanity check: should flag ONLY q11 and q31
flagged = [it["id"] for it in eval_set if is_roman_urdu(it["question"])]
print("flagged as Roman Urdu:", flagged)

from variants import RewriteStrictRAG

class RouterRAG(RewriteStrictRAG):
    """Rewrite only when the question looks Roman Urdu / Urdu script; otherwise search as-is."""
    name = "router"
    def prepare_query(self, question):
        return super().prepare_query(question) if is_roman_urdu(question) else question

router = RouterRAG()
base, strict = BaseRAG(), RewriteStrictRAG()

fmt = lambda r: "MISS" if r["first_rank"] is None else str(r["first_rank"])
for split in ("dev", "heldout"):
    items = [it for it in eval_set if it.get("split") == split]
    res = {n: score_retrieval(o, items, verbose=False) for n, o in
           [("base", base), ("strict", strict), ("router", router)]}
    print(f"\n=== {split} (n={len(res['base'][0])}) ===")
    for n, (rows, s) in res.items():
        print(f"{n:7}:", " | ".join(f"{k}={v:.2f}" for k, v in s.items()))

# diagnostic: does base best semantic distance separate hits from misses?
print("\nbase best semantic distance vs outcome (sorted):")
diag = []
for it in eval_set:
    if not it["expected_articles"]:
        continue
    groups, _ = base.retrieve(it["question"], base.detect(it["question"]))
    sem = [g.best_distance for g in groups if g.via == "semantic"]
    keys = [g.key for g in groups]
    ranks = [keys.index(a) + 1 for a in it["expected_articles"] if a in keys]
    diag.append((min(sem), it["id"], min(ranks) if ranks else None))
for d, qid, rank in sorted(diag, reverse=True):
    print(f"  {d:.3f}  {qid}  base_rank={'MISS' if rank is None else rank}")

In [ ]:
from variants import RewriteStrictRAG, RouterRAG, FusionRAG

systems = {"base": BaseRAG(), "strict": RewriteStrictRAG(),
           "router": RouterRAG(), "fusion": FusionRAG()}

for split in ("dev", "heldout"):
    items = [it for it in eval_set if it.get("split") == split]
    print(f"\n=== {split} ===")
    out = {n: score_retrieval(o, items, verbose=False) for n, o in systems.items()}
    for n, (rows, s) in out.items():
        print(f"{n:7}:", " | ".join(f"{k}={v:.2f}" for k, v in s.items()), f"(n={len(rows)})")
    fmt = lambda r: "MISS" if r["first_rank"] is None else str(r["first_rank"])
    print("  per question (base / strict / router / fusion):")
    for rows in zip(*[out[n][0] for n in systems]):
        ranks = [fmt(r) for r in rows]
        mark = "" if len(set(ranks)) == 1 else "  *"
        print(f"   {rows[0]['id']} {rows[0]['category']:14}", *[f"{x:>5}" for x in ranks], mark)

In [ ]:
def show_top(system, qid, n=6):
    it = next(i for i in eval_set if i["id"] == qid)
    groups, _ = system.retrieve(it["question"], system.detect(it["question"]))
    print(f"\n{qid}: {it['question']}  (expected {it['expected_articles']})")
    for g in groups[:n]:
        title = g.meta.get("article_title", "") if g.is_article else ""
        print(f"   {g.key:8} {title[:70]}")

for qid in ("q21", "q30", "q02"):
    for name in ("base", "fusion"):
        print(f"--- {name}", end="")
        show_top(systems[name], qid)

In [ ]:
FACTS = {
    "q01": ["An arrested person must be informed of the grounds for arrest as soon as may be",
            "An arrested person must be produced before a magistrate within 24 hours"],
    "q02": ["No person can be deprived of property except in accordance with law",
            "Compulsory acquisition requires a public purpose, authority of law and compensation"],
    "q03": ["The State must provide free and compulsory education to children aged five to sixteen"],
    "q04": ["There is no discrimination on the basis of sex",
            "The State may make special provisions for women and children"],
    "q05": ["No law may punish an act or omission that was not punishable by law when it was done"],
    "q10": ["Article 19 is about freedom of speech and expression, not religion (religion is Article 20)"],
    "q19": ["Article 96 has been omitted"],
    "q23": ["Islam is the State religion of Pakistan"],
    "q24": ["A person is entitled to a fair trial and due process"],
    "q25": ["No person accused of an offence can be compelled to be a witness against himself"],
    "q32": ["A Supreme Court judge holds office until the age of sixty-five"],
}
for it in eval_set:
    if it["id"] in FACTS:
        it["expected_facts"] = FACTS[it["id"]]
EVAL_PATH.write_text(json.dumps(eval_set, indent=2, ensure_ascii=False), encoding="utf-8")

JUDGE_PROMPT = """You are grading an answer about the Constitution of Pakistan.
For EACH expected fact, decide whether the answer states it (a paraphrase counts).
Return ONLY JSON: {"facts": [{"fact": "...", "verdict": "supported" | "missing" | "contradicted"}]}"""

DECLINE_PROMPT = """Decide whether the answer DECLINES to answer because the provided sources / the Constitution do not cover the question, WITHOUT stating an answer from outside knowledge.
Return ONLY JSON: {"declines": true | false, "reason": "<one short sentence>"}"""

def _judge(system_prompt, user_text):
    kwargs = dict(model=settings.deployment,
                  messages=[{"role": "system", "content": system_prompt},
                            {"role": "user", "content": user_text}],
                  max_completion_tokens=600)
    if settings.temperature is not None:
        kwargs["temperature"] = settings.temperature
    text = (client.chat.completions.create(**kwargs).choices[0].message.content or "").strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.M).strip()
    return json.loads(text)

def score_answers(system, items):
    rows = []
    for it in items:
        facts, decline = it.get("expected_facts"), it.get("should_decline")
        if not facts and not decline:
            continue
        r = system.ask(it["question"])
        row = {"id": it["id"], "answer": r.answer, "error": r.error, "decline_expected": bool(decline)}
        try:
            if decline:
                j = _judge(DECLINE_PROMPT, f"Question: {it['question']}\n\nAnswer:\n{r.answer}")
                row["declines"], row["note"] = bool(j["declines"]), j.get("reason", "")
            else:
                j = _judge(JUDGE_PROMPT, f"Question: {it['question']}\n\nAnswer:\n{r.answer}\n\n"
                                         f"Expected facts:\n{json.dumps(facts)}")
                row["verdicts"] = [f["verdict"] for f in j["facts"]]
                if len(row["verdicts"]) != len(facts):
                    row["judge_error"] = "fact count mismatch"
        except Exception as e:
            row["judge_error"] = f"{type(e).__name__}: {str(e)[:100]}"
        rows.append(row)
    return rows

def summarize(name, rows):
    fr = [r for r in rows if "verdicts" in r and "judge_error" not in r]
    v = [x for r in fr for x in r["verdicts"]]
    dr = [r for r in rows if r["decline_expected"] and "declines" in r]
    errs = sum(1 for r in rows if "judge_error" in r or r["error"])
    print(f"{name:7}: facts supported {v.count('supported')}/{len(v)} | "
          f"contradicted {v.count('contradicted')} | "
          f"declines correctly {sum(r['declines'] for r in dr)}/{len(dr)} | errors {errs}")

targets = [it for it in eval_set if it.get("expected_facts") or it.get("should_decline")]
results = {n: score_answers(systems[n], targets) for n in ("base", "fusion")}

for n, rows in results.items():
    summarize(n, rows)

print("\nper question (S=supported M=missing C=contradicted; decline: Y/N):")
code = {"supported": "S", "missing": "M", "contradicted": "C"}
for b, f in zip(results["base"], results["fusion"]):
    show = lambda r: ("decline=" + ("Y" if r.get("declines") else "N")) if r["decline_expected"] \
        else "".join(code.get(x, "?") for x in r.get("verdicts", ["?"]))
    print(f"  {b['id']}  base={show(b):10} fusion={show(f)}")

print("\nanswers that were not fully supported / wrongly answered:")
for n, rows in results.items():
    for r in rows:
        bad = (r["decline_expected"] and not r.get("declines")) or \
              (not r["decline_expected"] and any(x != "supported" for x in r.get("verdicts", [])))
        if bad or "judge_error" in r:
            print(f"\n[{n}] {r['id']} {r.get('judge_error', '')}\n  {r['answer'][:350]}")

In [ ]:
# 1) the footnotes for Article 175A (page-level, may include neighbouring articles)
res = store.get_article("175A")
seen = set()
for cid, m in zip(res["ids"], res["metadatas"]):
    n = m.get("amendment_notes")
    if n and n not in seen:
        seen.add(n)
        print(f"{cid} (pdf p.{m.get('pdf_page')}):\n{n}\n")

# 2) save this answer-eval run
out_dir = Path("eval/results"); out_dir.mkdir(parents=True, exist_ok=True)
stamp = time.strftime("%Y%m%d-%H%M%S")
path = out_dir / f"answers-{stamp}.json"
path.write_text(json.dumps({"model": settings.deployment, "temperature": settings.temperature,
                            "systems": list(results), "results": results},
                           indent=2, ensure_ascii=False), encoding="utf-8")
print("saved", path)

In [ ]:
q35 = {
    "id": "q35",
    "question": "When was Article 175A inserted?",
    "expected_articles": ["175A"],
    "category": "amendment_history",
    "should_decline": False,
    "verified": True,
    "split": "dev",
    "expected_facts": [
        "Article 175A was originally inserted by the Eighteenth Amendment in 2010",
        "It was later substantially amended by the Twenty-sixth Amendment (2024) and/or the Twenty-seventh Amendment (2025)",
    ],
}
if not any(it["id"] == "q35" for it in eval_set):
    eval_set.append(q35)
    EVAL_PATH.write_text(json.dumps(eval_set, indent=2, ensure_ascii=False), encoding="utf-8")

only_q35 = [it for it in eval_set if it["id"] == "q35"]
for name in ("base", "fusion"):
    rows = score_answers(systems[name], only_q35)
    print(f"\n[{name}] verdicts={rows[0].get('verdicts')}  judge_error={rows[0].get('judge_error')}")
    print(rows[0]["answer"])

In [ ]:
JUDGE_PROMPT = """You are grading an answer about the Constitution of Pakistan.
For EACH expected fact, decide:
- "supported": the answer EXPLICITLY states the fact (a paraphrase counts).
- "missing": the answer does not mention it. Absence is NEVER "supported".
- "contradicted": the answer states something incompatible with it.
Return ONLY JSON: {"facts": [{"fact": "...", "verdict": "supported" | "missing" | "contradicted"}]}"""

from variants import FusionNotesRAG

class NotesOnlyRAG(FusionNotesRAG):
    name = "fusion_notes_norule"
    use_rule = False

contenders = {"base": systems["base"], "fusion": systems["fusion"],
              "notes_norule": NotesOnlyRAG(), "notes+rule": FusionNotesRAG()}

q35_item = [it for it in eval_set if it["id"] == "q35"]
for name, sysobj in contenders.items():
    print(f"\n===== {name} (3 runs) =====")
    for run in range(3):
        row = score_answers(sysobj, q35_item)[0]
        print(f"run {run+1}: verdicts={row.get('verdicts')} | {row['answer'][:230]!r}")

# size of the notes block for 175A before/after cleanup
res = store.get_article("175A")
raw = {c["amendment_notes"] for c in res["metadatas"] if c.get("amendment_notes")}

def clean_notes():
    raise NotImplementedError

def clean_notes(chunks):
    notes, seen = [], set()

    for chunk in sorted(chunks, key=lambda c: c.meta.get("chunk_index", 0)):
        raw_note = chunk.meta.get("amendment_notes") or ""
        for note in raw_note.split("|"):
            note = re.sub(r"\s+", " ", note).strip()
            if note and note not in seen:
                seen.add(note)
                notes.append(note)

    # Put notes with identifiable years in chronological order.
    def year_key(note):
        match = re.search(r"\b(?:19|20)\d{2}\b", note)
        return int(match.group()) if match else float("inf")

    return sorted(notes, key=year_key)

items = clean_notes([
    Chunk(i, d, m)
    for i, d, m in zip(res["ids"], res["documents"], res["metadatas"])
])
print("\nnotes chars raw:", sum(len(r) for r in raw), "| cleaned:", sum(len(t) for t in items), "| items:", len(items))
print("first 6 cleaned:", items[:6])

In [ ]:
from variants import clean_notes
from base_rag import Chunk

res = store.get_article("175A")
raw = {m["amendment_notes"] for m in res["metadatas"] if m.get("amendment_notes")}
items = clean_notes([Chunk(i, d, m) for i, d, m in zip(res["ids"], res["documents"], res["metadatas"])])

print("notes chars raw:", sum(len(r) for r in raw), "| cleaned:", sum(len(t) for t in items), "| items:", len(items))
print("\nfirst 8 cleaned (oldest first):")
for t in items[:8]:
    print(" -", t)

In [ ]:
q35_item = [it for it in eval_set if it["id"] == "q35"]
for name, sysobj in contenders.items():
    print(f"\n===== {name} (3 runs) =====")
    for run in range(3):
        row = score_answers(sysobj, q35_item)[0]
        print(f"run {run+1}: verdicts={row.get('verdicts')} | {row['answer'][:260]!r}")

In [ ]:
# 1) the fact now names the Act number and the Eighteenth Amendment explicitly
for it in eval_set:
    if it["id"] == "q35":
        it["expected_facts"] = [
            "Article 175A was originally inserted in 2010 (by Act No. 10 of 2010, i.e. the Eighteenth Amendment)",
            "It was later amended by the Twenty-sixth Amendment (2024) and/or the Twenty-seventh Amendment (2025)",
        ]
EVAL_PATH.write_text(json.dumps(eval_set, indent=2, ensure_ascii=False), encoding="utf-8")

# 2) re-run 5x each and keep the answers
q35_item = [it for it in eval_set if it["id"] == "q35"]
runs = {}
for name in ("base", "notes+rule"):
    runs[name] = [score_answers(contenders[name], q35_item)[0] for _ in range(5)]
    v = [r.get("verdicts") for r in runs[name]]
    first_ok = sum(1 for x in v if x and x[0] == "supported")
    print(f"\n{name}: first fact supported in {first_ok}/5 runs")
    for i, r in enumerate(runs[name], 1):
        print(f"  run {i}: {r.get('verdicts')} | {r['answer'][:160]!r}")

# 3) candidate articles for fresh amendment-history questions: print their footnotes
for num in ("10A", "19A", "25A"):
    res = store.get_article(num)
    print(f"\n=== Article {num} ===")
    shown = set()
    for m in res["metadatas"]:
        n = m.get("amendment_notes")
        if n and n not in shown:
            shown.add(n)
            print(n[:900])

In [ ]:
NEW_AMEND = [
    ("q36", "When was Article 10A inserted into the Constitution?",
     ["Article 10A was inserted by the Eighteenth Amendment in 2010"]),
    ("q37", "Which amendment added Article 19A on the right to information?",
     ["Article 19A was inserted by the Eighteenth Amendment in 2010"]),
    ("q38", "How did Article 25A come to be part of the Constitution?",
     ["Article 25A was inserted by the Eighteenth Amendment in 2010"]),
]
have = {it["id"] for it in eval_set}
for qid, q, facts in NEW_AMEND:
    if qid not in have:
        art = {"q36": "10A", "q37": "19A", "q38": "25A"}[qid]
        eval_set.append({"id": qid, "question": q, "expected_articles": [art],
                         "category": "amendment_history", "should_decline": False,
                         "verified": True, "split": "heldout", "expected_facts": facts})
EVAL_PATH.write_text(json.dumps(eval_set, indent=2, ensure_ascii=False), encoding="utf-8")

new_items = [it for it in eval_set if it["id"] in ("q36", "q37", "q38")]
for name in ("base", "notes+rule"):
    print(f"\n===== {name} (5 runs each) =====")
    for it in new_items:
        rows = [score_answers(contenders[name], [it])[0] for _ in range(5)]
        ok = sum(1 for r in rows if r.get("verdicts") == ["supported"])
        print(f"{it['id']}: fact supported {ok}/5")
        for r in rows[:2]:
            print(f"     e.g. {r['answer'][:170]!r}")

In [ ]:
import importlib, variants
importlib.reload(variants)
from variants import InsertionNotesRAG, explicit_insertion_notes
print("ok")

In [ ]:
from variants import InsertionNotesRAG, explicit_insertion_notes

ins = InsertionNotesRAG()

# 1) no LLM: does the extractor find the right notes for each article?
for num in ("175A", "10A", "19A", "25A"):
    res = store.get_article(num)
    ch = [Chunk(i, d, m) for i, d, m in zip(res["ids"], res["documents"], res["metadatas"])]
    print(num, "->", explicit_insertion_notes(num, ch))

# 2) answers: base vs the new variant, 5 runs each on q35-q38
items = [it for it in eval_set if it["id"] in ("q35", "q36", "q37", "q38")]
for name, sysobj in [("base", systems["base"]), ("fusion_insertion", ins)]:
    print(f"\n===== {name} =====")
    for it in items:
        rows = [score_answers(sysobj, [it])[0] for _ in range(5)]
        ok = sum(1 for r in rows if r.get("verdicts") and r["verdicts"][0] == "supported")
        print(f"{it['id']}: first fact supported {ok}/5 | e.g. {rows[0]['answer'][:150]!r}")

# 3) candidate FRESH articles (no LLM): which articles have a 'New Article ... ins.' footnote?
res = store.collection.get(where={"kind": "article"}, include=["metadatas"])
found = {}
for m in res["metadatas"]:
    n = m.get("amendment_notes") or ""
    if re.search(r"New\s+Article\s+\d+\s*[A-Z]*\s+ins", n, re.I):
        found[m["article_number"]] = n
print("\narticles with an explicit 'New Article ... ins.' note:", len(found))
print(sorted(found, key=lambda a: (int(re.match(r'\d+', a).group()), a))[:40])

In [ ]:
fresh = ["2A", "11", "24", "140A", "203F"]
for num in fresh:
    res = store.get_article(num)
    ch = [Chunk(i, d, m) for i, d, m in zip(res["ids"], res["documents"], res["metadatas"])]
    title = res["metadatas"][0].get("article_title")
    print(f"\n=== Article {num}: {title!r}")
    print("  explicit:", explicit_insertion_notes(num, ch))
    shown = set()
    for m in res["metadatas"]:
        n = m.get("amendment_notes")
        if n and n not in shown:
            shown.add(n)
            print("  all notes:", n[:600])

In [ ]:
FRESH = [
    ("q39", "How did Article 2A come to be part of the Constitution?", "2A", "pos",
     ["Article 2A was inserted by the Revival of the Constitution of 1973 Order, 1985 (P.O. No. 14 of 1985)"]),
    ("q40", "When was Article 140A on local government added to the Constitution?", "140A", "pos",
     ["The current Article 140A was inserted by the Eighteenth Amendment in 2010"]),
    ("q41", "When was Article 11 inserted into the Constitution?", "11", "neg", None),
    ("q42", "Which amendment inserted Article 24?", "24", "neg", None),
    ("q43", "When did Article 140A become part of the Constitution?", "140A", "pos",
     ["The current Article 140A was inserted by the Eighteenth Amendment in 2010"]),
]
have = {it["id"] for it in eval_set}
for qid, q, art, kind, facts in FRESH:
    if qid in have:
        continue
    item = {"id": qid, "question": q, "category": "amendment_history",
            "verified": True, "split": "heldout2"}
    if kind == "pos":
        item.update(expected_articles=[art], should_decline=False, expected_facts=facts)
    else:
        item.update(expected_articles=[], should_decline=True)   # correct behaviour: sources don't say
    eval_set.append(item)
EVAL_PATH.write_text(json.dumps(eval_set, indent=2, ensure_ascii=False), encoding="utf-8")

# no LLM: what does the model actually see for Article 11?
q = "When was Article 11 inserted into the Constitution?"
groups, _ = ins.retrieve(q, ins.detect(q))
ctx, _ = ins.build_context(q, groups)
i = ctx.find("Amendment notes")
print("context for Article 11, notes part:\n", ctx[i:i + 500] if i >= 0 else "(no notes block in context)")
print("explicit-insertion section present:", "Explicit insertion footnotes" in ctx)

# answers
RUNS = 5
fresh_items = [it for it in eval_set if it.get("split") == "heldout2"]
for name, sysobj in [("base", systems["base"]), ("fusion_insertion", ins)]:
    print(f"\n===== {name} =====")
    for it in fresh_items:
        rows = [score_answers(sysobj, [it])[0] for _ in range(RUNS)]
        if it["should_decline"]:
            good = [r for r in rows if r.get("declines")]
            label = "declines"
        else:
            good = [r for r in rows if r.get("verdicts") and all(v == "supported" for v in r["verdicts"])]
            label = "facts supported"
        print(f"{it['id']}: {label} {len(good)}/{RUNS}")
        bad = next((r for r in rows if r not in good), None)
        if bad:
            print("     failing example:", bad["answer"][:230].replace("\n", " "))

In [ ]:
from variants import InsertionNotesV2RAG
from variants import InsertionNotesRAG
ins = InsertionNotesRAG()
v2 = InsertionNotesV2RAG()
RUNS = 3
items = [it for it in eval_set if it["id"] in ("q35","q36","q37","q38","q39","q40","q41","q42","q43")]
print("system: fusion_insertion (v1) vs fusion_insertion_v2")
for name, sysobj in [("v1", ins), ("v2", v2)]:
    print(f"\n===== {name} =====")
    for it in items:
        rows = [score_answers(sysobj, [it])[0] for _ in range(RUNS)]
        if it["should_decline"]:
            good = [r for r in rows if r.get("declines")]
        else:
            good = [r for r in rows if r.get("verdicts") and all(v == "supported" for v in r["verdicts"])]
        print(f"{it['id']}: {len(good)}/{RUNS}", end="")
        bad = next((r for r in rows if r not in good), None)
        print(f"   | failing: {bad['answer'][:170]!r}" if bad else "")

In [ ]:
from evaluation import score_answers, summarize
from base_rag import Chunk
from variants import explicit_insertion_notes

# ---- A) regression on non-amendment questions ----
reg_items = [it for it in eval_set
             if (it.get("expected_facts") or it.get("should_decline"))
             and it["category"] != "amendment_history"]
print(len(reg_items), "regression questions\n")

reg = {}
for name, sysobj in [("fusion", systems["fusion"]), ("fusion_insertion_v2", v2)]:
    reg[name] = score_answers(sysobj, reg_items)
    summarize(name, reg[name])

code = {"supported": "S", "missing": "M", "contradicted": "C"}
show = lambda r: ("decline=" + ("Y" if r.get("declines") else "N")) if r["decline_expected"] \
    else "".join(code.get(x, "?") for x in r.get("verdicts", ["?"]))
print("\nper question where the two systems differ:")
for a, b in zip(reg["fusion"], reg["fusion_insertion_v2"]):
    if show(a) != show(b):
        print(f"  {a['id']}  fusion={show(a)}  v2={show(b)}")
        print("     fusion:", a["answer"][:200].replace("\n", " "))
        print("     v2    :", b["answer"][:200].replace("\n", " "))

# ---- B) fresh candidates: articles with their OWN explicit insertion note (no LLM) ----
used = {"175A", "10A", "19A", "25A", "2A", "140A", "11", "24"}
col = store.collection.get(where={"kind": "article"}, include=["metadatas"])
keys = sorted({m["article_number"] for m in col["metadatas"]},
              key=lambda a: (int(re.match(r"\d+", a).group()), a))
own = []
for k in keys:
    if k in used:
        continue
    r_ = store.get_article(k)
    ch = [Chunk(i, d, m) for i, d, m in zip(r_["ids"], r_["documents"], r_["metadatas"])]
    ex = explicit_insertion_notes(k, ch)
    if ex:
        own.append((k, ex))
print(f"\n{len(own)} unused articles with their own explicit insertion note:")
for k, ex in own:
    print(f"  {k:7} -> {ex[-1][:120]}")

In [ ]:
FRESH2 = [
    ("q44", "When was Article 203GG inserted?", "203GG",
     ["Article 203GG was inserted by the Constitution (Second Amendment) Order, 1982 (P.O. No. 5 of 1982)"]),
    ("q45", "Which law inserted Article 224A?", "224A",
     ["Article 224A was inserted by Act No. 5 of 2012"]),
]
have = {it["id"] for it in eval_set}
for qid, q, art, facts in FRESH2:
    if qid not in have:
        eval_set.append({"id": qid, "question": q, "expected_articles": [art],
                         "category": "amendment_history", "should_decline": False,
                         "verified": True, "split": "heldout3", "expected_facts": facts})
EVAL_PATH.write_text(json.dumps(eval_set, indent=2, ensure_ascii=False), encoding="utf-8")

RUNS = 5
items = [it for it in eval_set if it["id"] in ("q44", "q45")]
for name, sysobj in [("base", systems["base"]), ("fusion_insertion_v2", v2)]:
    print(f"\n===== {name} =====")
    for it in items:
        rows = [score_answers(sysobj, [it])[0] for _ in range(RUNS)]
        good = [r for r in rows if r.get("verdicts") and all(v == "supported" for v in r["verdicts"])]
        print(f"{it['id']}: {len(good)}/{RUNS}")
        bad = next((r for r in rows if r not in good), None)
        print(f"   failing: {bad['answer'][:230]!r}" if bad else "   all runs correct")

# q46: read, don't score
print("\n===== q46 (read these yourself) =====")
for name, sysobj in [("base", systems["base"]), ("v2", v2)]:
    r = sysobj.ask("When was Article 203F amended?")
    print(f"[{name}] {r.answer[:400]}\n")

In [ ]:
from variants import DeclineToneRAG
from evaluation import score_answers, summarize

tone = DeclineToneRAG()
assert "at most two plain sentences" in tone.system_prompt

dec = [it for it in eval_set if it.get("should_decline") and it["category"] == "out_of_scope"]
print("decline questions:", [it["id"] for it in dec])

for name, sysobj in [("fusion", systems["fusion"]), ("fusion_tone", tone)]:
    print(f"\n===== {name} =====")
    for it in dec:
        r = sysobj.ask(it["question"])
        print(f"{it['id']} | citations={len(r.cited_articles)} words={len(r.answer.split())}")
        print("   ", r.answer.replace("\n", " ")[:300])

# regression on everything that should be ANSWERED (over-decline check)
ans = [it for it in eval_set if it.get("expected_facts") and it["category"] != "amendment_history"]
print(f"\nregression on {len(ans)} answerable questions")
for name, sysobj in [("fusion", systems["fusion"]), ("fusion_tone", tone)]:
    rows = score_answers(sysobj, ans)
    summarize(name, rows)
    declined = [r["id"] for r in rows if any(v == "missing" for v in r.get("verdicts", []))]
    print("   questions with a missing fact:", declined)

In [ ]:
TONE_RULE_V2 = """
- If the question cannot be answered from the text of the Constitution at all (for example: who currently holds an office, current events, tax rates, personal legal advice), reply in one or two plain sentences: say you can only answer from the text of the Constitution of Pakistan and that it does not cover this. Do not cite any article for such a question and do not list related provisions.
- If the sources answer only part of the question, answer that part normally and say what is not covered."""

class DeclineToneV2RAG(FusionRAG):
    name = "fusion_tone_v2"
    system_prompt = FusionRAG.system_prompt + TONE_RULE_V2

tone2 = DeclineToneV2RAG()

# 1) decline answers
for it in dec:
    r = tone2.ask(it["question"])
    print(f"{it['id']} | citations={len(r.cited_articles)} words={len(r.answer.split())} | {r.answer[:220]!r}")

# 2) is q01's missing fact tone-related or run-to-run noise?
q01 = [it for it in eval_set if it["id"] == "q01"]
for name, sysobj in [("fusion", systems["fusion"]), ("fusion_tone", tone), ("fusion_tone_v2", tone2)]:
    rows = [score_answers(sysobj, q01)[0] for _ in range(5)]
    ok = sum(1 for r in rows if r.get("verdicts") and all(v == "supported" for v in r["verdicts"]))
    print(f"{name:15}: q01 both facts supported {ok}/5")

In [ ]:
for name, sysobj in [("fusion", systems["fusion"]), ("fusion_tone", tone)]:
    r = sysobj.ask("Can the police arrest me without telling me why?")
    print(f"[{name}] {len(r.answer.split())} words\n{r.answer}\n")

In [ ]:
from variants import ScopeGateRAG
gate = ScopeGateRAG()

# 1) false-decline check: every question that SHOULD be answered
bad = [it["id"] for it in eval_set if not it.get("should_decline") and not gate.in_scope(it["question"])]
print("answerable questions wrongly flagged out-of-scope:", bad)

# 2) out-of-scope questions should be flagged
print("decline questions flagged out-of-scope:",
      {it["id"]: (not gate.in_scope(it["question"])) for it in eval_set if it.get("should_decline") and it["category"] == "out_of_scope"})

# 3) a few borderline questions (my guess of the right label in the comment, not in the data)
for q in ["Who was the first President of Pakistan?",                  # probably out (history)
          "What are the qualifications to become Prime Minister?",     # in
          "Can the Prime Minister dissolve the National Assembly?",    # in
          "What is the capital of Pakistan?",                          # unclear
          "Is a divorce legal under the Constitution?",                # borderline
          "Mera bijli ka bill zyada aya, kya karun?"]:                 # out (Roman Urdu, unrelated)
    print(f"{gate.in_scope(q)!s:6} <- {q}")

# 4) the decline replies, and the 24-hour rule survives
for it in [i for i in eval_set if i.get("should_decline") and i["category"] == "out_of_scope"]:
    r = gate.ask(it["question"])
    print(it["id"], "|", len(r.cited_articles), "citations |", r.answer[:120])
q01 = [it for it in eval_set if it["id"] == "q01"]
rows = [score_answers(gate, q01)[0] for _ in range(5)]
print("q01 both facts:", sum(1 for r in rows if r.get("verdicts") and all(v == "supported" for v in r["verdicts"])), "/5")

In [ ]:
from variants import ScopeGateRAG
gate = ScopeGateRAG()

bad = [it["id"] for it in eval_set if not it.get("should_decline") and not gate.in_scope(it["question"])]
print("answerable questions wrongly flagged out-of-scope:", bad)
print("decline questions flagged:",
      {it["id"]: (not gate.in_scope(it["question"])) for it in eval_set
       if it.get("should_decline") and it["category"] == "out_of_scope"})

# q01: interleaved, with a judge-free check
q01 = [it for it in eval_set if it["id"] == "q01"]
for run in range(5):
    for name, sysobj in [("fusion", systems["fusion"]), ("gate", gate)]:
        row = score_answers(sysobj, q01)[0]
        a = row["answer"]
        print(f"run {run+1} {name:6} judge={row.get('verdicts')} words={len(a.split()):3} "
              f"mentions '24 hours'={'24 hours' in a}")

In [ ]:
from variants import CompleteRAG
comp = CompleteRAG()
assert "separate clauses" in comp.system_prompt

q = "Can the police arrest me without telling me why?"
for name, sysobj in [("fusion", systems["fusion"]), ("complete", comp)]:
    outs = [sysobj.ask(q).answer for _ in range(10)]
    has24 = sum("24 hours" in a for a in outs)
    has_prev = sum("preventive" in a.lower() for a in outs)
    words = sum(len(a.split()) for a in outs) / len(outs)
    print(f"{name:9}: '24 hours' in {has24}/10 | 'preventive' in {has_prev}/10 | avg {words:.0f} words")
print("\nexample (complete):\n", outs[0])

In [ ]:
col = store.collection.get(where={"kind": "article"}, include=["documents", "metadatas"])
cands = []
for d, m in zip(col["documents"], col["metadatas"]):
    if m.get("chunk_count", 1) == 1 and "Provided that" in d and not m.get("omitted") and 300 < len(d) < 1100:
        n = int(re.match(r"\d+", m["article_number"]).group())
        cands.append((n, m["article_number"], m.get("article_title"), d))
cands.sort()
step = max(1, len(cands) // 10)
for _, k, t, d in cands[::step][:10]:
    print(f"\n=== Article {k}: {t}\n{d.split(chr(10)+chr(10), 1)[-1][:650]}")
print(f"\n({len(cands)} candidates)")

In [ ]:
q = "Can the police arrest me without telling me why?"
for name, sysobj in [("fusion", systems["fusion"]), ("complete", comp)]:
    ok, errs = [], []
    for _ in range(10):
        r = sysobj.ask(q)
        (errs if r.error else ok).append(r)
    print(f"{name:9}: {len(ok)} ok, {len(errs)} errors"
          f" | '24 hours' in {sum('24 hours' in r.answer for r in ok)}/{len(ok)}"
          f" | avg {sum(len(r.answer.split()) for r in ok)/max(len(ok),1):.0f} words")
    if errs:
        print("   first error:", errs[0].error[:300])

In [ ]:
from variants import CompleteRAG

FRESH3 = [
    ("q47", "How long can a President serve?", "44",
     ["The President's term is five years",
      "No person may hold the office of President for more than two consecutive terms"]),
    ("q48", "At what age does a Federal Constitutional Court judge retire?", "175I",
     ["A Federal Constitutional Court judge holds office until the age of sixty-eight",
      "The Chief Justice of the Federal Constitutional Court serves a term of three years"]),
    ("q49", "Can a Province hand over functions to the Federal Government?", "147",
     ["Yes, with the consent of the Federal Government",
      "The Provincial Government must get the functions ratified by the Provincial Assembly within sixty days"]),
]
have = {it["id"] for it in eval_set}
for qid, q, art, facts in FRESH3:
    if qid not in have:
        eval_set.append({"id": qid, "question": q, "expected_articles": [art],
                         "category": "completeness", "should_decline": False,
                         "verified": True, "split": "heldout4", "expected_facts": facts})
EVAL_PATH.write_text(json.dumps(eval_set, indent=2, ensure_ascii=False), encoding="utf-8")

items = [it for it in eval_set if it["id"] in ("q47", "q48", "q49")]
RUNS = 6
for name, sysobj in [("fusion", systems["fusion"]), ("fusion_complete", CompleteRAG())]:
    print(f"\n===== {name} =====")
    for it in items:
        main = extra = valid = 0
        for _ in range(RUNS):
            row = score_answers(sysobj, [it])[0]
            if row.get("judge_error") or row.get("error"):
                continue
            valid += 1
            main += row["verdicts"][0] == "supported"
            extra += row["verdicts"][1] == "supported"
        print(f"{it['id']}: main fact {main}/{valid} | proviso/extra fact {extra}/{valid}")

In [ ]:
import chromadb
coll = chromadb.PersistentClient(path="./chroma_db_v2").get_collection("pakistan_constitution")
for art in ("2A", "25A", "140A"):
    r = coll.get(where={"article_number": art}, include=["metadatas"])
    for i, m in zip(r["ids"], r["metadatas"]):
        print(i, "| article_notes:", m.get("article_notes"))
        print("     notes:", (m.get("amendment_notes") or "")[:300])

In [ ]:
import chromadb
coll = chromadb.PersistentClient(path="./chroma_db_v2").get_collection("pakistan_constitution")
for art in ("2A", "25A", "140A"):
    r = coll.get(where={"article_number": art}, include=["metadatas"])
    print(art, "|", (r["metadatas"][0].get("article_notes") or "")[:230])

In [ ]:
from pathlib import Path
from variants import InsertionNotesV2RAG

class Probe(InsertionNotesV2RAG):
    cache_path = Path("logs/probe_rewrite_cache.json")

s = Probe(n_results=20, max_groups=12, log_path=None)
for q in ["what are the rights of minorities in pakistan", "tell me about the right of women"]:
    groups, _ = s.retrieve(q, s.detect(q))
    print(q, "->", [g.key for g in groups])

In [ ]:
import json
from evaluation import load_eval, EVAL_PATH
from variants import InsertionNotesV2RAG, BroadRAG, BROAD_RE

items = load_eval()

# 1) false positives: which existing questions would the trigger catch?
hits = [(it["id"], it["question"]) for it in items
        if BROAD_RE.search(it["question"]) and "article" not in it["question"].lower()]
print("existing questions matching BROAD_RE:", hits or "none")

# 2) add fresh broad questions (retrieval-only; expected = articles that should be in the context)
NEW = [
    ("q50", "What are the rights of children?", ["25A", "11"]),
    ("q51", "What does the Constitution say about education?", ["25A", "37"]),
    ("q52", "What are the rights of an accused person?", ["10", "10A", "13"]),
    ("q53", "Tell me about freedom of expression and assembly", ["19", "16"]),
    ("q54", "What does the Constitution say about religious freedom?", ["20"]),
    ("q55", "what are the rights of minorities in pakistan", ["36", "20"]),     # seen before: dev
    ("q56", "tell me about the right of women", ["25", "34"]),                  # seen before: dev
]
have = {it["id"] for it in items}
for qid, q, exp in NEW:
    if qid not in have:
        items.append({"id": qid, "question": q, "expected_articles": exp, "category": "broad",
                      "should_decline": False, "verified": False,
                      "split": "dev" if qid in ("q55", "q56") else "heldout5"})
EVAL_PATH.write_text(json.dumps(items, indent=2, ensure_ascii=False), encoding="utf-8")

base, broad = InsertionNotesV2RAG(log_path=None), BroadRAG(log_path=None)
store = base.store
print("\nverify the expected articles:")
for qid, q, exp in NEW:
    for a in exp:
        r = store.get_article(a)
        print(f"  {qid} {a}: {r['metadatas'][0].get('article_title') if r['ids'] else 'NOT FOUND'}")

# 3) recall and context size, default vs broad
print(f"\n{'':4}{'default recall':>16}{'broad recall':>14}{'chars default':>15}{'chars broad':>13}")
for it in [i for i in items if i["category"] == "broad"]:
    row = []
    for s in (base, broad):
        groups, _ = s.retrieve(it["question"], s.detect(it["question"]))
        keys = [g.key for g in groups]
        rec = sum(a in keys for a in it["expected_articles"]) / len(it["expected_articles"])
        ctx, _ = s.build_context(it["question"], groups)
        row += [rec, len(ctx)]
    print(f"{it['id']:4}{row[0]:>16.2f}{row[2]:>14.2f}{row[1]:>15}{row[3]:>13}")

In [ ]:
import statistics as stats
from variants import InsertionNotesV2RAG, BroadRAG

base = InsertionNotesV2RAG(log_path=None)
broad = BroadRAG(store=base.store, log_path=None)

QS = {"what are the rights of minorities in pakistan": ["36", "20"],
      "tell me about the right of women": ["25", "34"]}
for q, want in QS.items():
    print("\n", q)
    for name, s in (("default", base), ("broad", broad)):
        ok = [r for r in (s.ask(q) for _ in range(4)) if not r.error]
        cited = {a: sum(a in r.cited_articles for r in ok) for a in want}
        print(f"  {name:8} cited {cited} of {len(ok)} runs | "
              f"{stats.mean(len(r.answer.split()) for r in ok):.0f} words | "
              f"{stats.mean(r.usage['total_tokens'] for r in ok):.0f} tokens | "
              f"{stats.mean(r.timings['total_s'] for r in ok):.1f}s")